# Data Quality Framework

Este notebook centraliza as regras de validação, premissas de limpeza e definições de esquemas (Data Contracts) utilizados em todo o pipeline, garantindo consistência e transparência sobre a qualidade dos dados.

## 1. Definições de Esquemas (Schema Enforcement)

Utilizamos `StructType` do Spark para forçar a tipagem correta na camada Silver, garantindo que mudanças na fonte não quebrem o consumo analítico.

In [0]:
from pyspark.sql.types import *

# Esquema Oficial: Clientes Silver
SCHEMA_CLIENTES_SILVER = StructType([
    StructField("customer_id", StringType(), False),
    StructField("nome_cliente", StringType(), True),
    StructField("segmento", StringType(), True),
    StructField("porte", StringType(), True),
    StructField("cidade", StringType(), True),
    StructField("estado", StringType(), True),
    StructField("data_cadastro", DateType(), True),
    StructField("email", StringType(), True),
    StructField("flg_ativo", BooleanType(), True),
    StructField("flg_qualidade", BooleanType(), True),
    StructField("_silver_processed_at", TimestampType(), True)
])

# Esquema Oficial: Produtos Silver
SCHEMA_PRODUTOS_SILVER = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("subcategory", StringType(), True),
    StructField("status", StringType(), True),
    StructField("list_price", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("family", StringType(), True),
    StructField("tags", StringType(), True),
    StructField("updated_at", TimestampType(), True),
    StructField("flg_qualidade", BooleanType(), True),
    StructField("_silver_processed_at", TimestampType(), True)
])

# Esquema Oficial: Pedidos Cabeçalho Silver
SCHEMA_PEDIDOS_SILVER = StructType([
    StructField("pedido_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("vendedor_id", StringType(), True),
    StructField("data_pedido", DateType(), True),
    StructField("data_previsao_entrega", DateType(), True),
    StructField("status_pedido", StringType(), True),
    StructField("valor_bruto", DoubleType(), True),
    StructField("valor_desconto", DoubleType(), True),
    StructField("valor_liquido", DoubleType(), True),
    StructField("prioridade", StringType(), True),
    StructField("origem_pedido", StringType(), True),
    StructField("data_atualizacao", TimestampType(), True),
    StructField("flg_qualidade", BooleanType(), True),
    StructField("_silver_processed_at", TimestampType(), True)
])

# Esquema Oficial: Pedidos Itens Silver
SCHEMA_PEDIDOS_ITENS_SILVER = StructType([
    StructField("pedido_id", StringType(), False),
    StructField("item_id", IntegerType(), False),
    StructField("product_id", StringType(), True),
    StructField("quantidade", IntegerType(), True),
    StructField("preco_unitario", DoubleType(), True),
    StructField("valor_item", DoubleType(), True),
    StructField("status_item", StringType(), True),
    StructField("flg_qualidade", BooleanType(), True),
    StructField("_silver_processed_at", TimestampType(), True)
])

# Esquema Oficial: Regiões Silver
SCHEMA_REGIOES_SILVER = StructType([
    StructField("regional_code", StringType(), False),
    StructField("regional_name", StringType(), True),
    StructField("state", StringType(), True),
    StructField("manager_name", StringType(), True),
    StructField("flg_ativo", BooleanType(), True),
    StructField("flg_qualidade", BooleanType(), True),
    StructField("_silver_processed_at", TimestampType(), True)
])

# Esquema Oficial: Canais Silver
SCHEMA_CANAIS_SILVER = StructType([
    StructField("id_canal", StringType(), False),
    StructField("nome_canal", StringType(), True),
    StructField("tipo_canal", StringType(), True),
    StructField("flg_ativo", BooleanType(), True),
    StructField("observacao", StringType(), True),
    StructField("flg_qualidade", BooleanType(), True),
    StructField("_silver_processed_at", TimestampType(), True)
])

# Esquema Oficial: Vendedores Silver
SCHEMA_VENDEDORES_SILVER = StructType([
    StructField("seller_id", StringType(), False),
    StructField("seller_name", StringType(), True),
    StructField("canal_id", StringType(), True),
    StructField("regional_code", StringType(), True),
    StructField("hire_date", DateType(), True),
    StructField("flg_ativo", BooleanType(), True),
    StructField("flg_qualidade", BooleanType(), True),
    StructField("_silver_processed_at", TimestampType(), True)
])

# Esquema Oficial: Entregas Silver
SCHEMA_ENTREGAS_SILVER = StructType([
    StructField("delivery_id", StringType(), False),
    StructField("order_id", StringType(), True),
    StructField("carrier_name", StringType(), True),
    StructField("carrier_mode", StringType(), True),
    StructField("delivery_status", StringType(), True),
    StructField("shipped_at", TimestampType(), True),
    StructField("delivered_at", TimestampType(), True),
    StructField("destination_state", StringType(), True),
    StructField("destination_city", StringType(), True),
    StructField("cost", DoubleType(), True),
    StructField("flg_qualidade", BooleanType(), True),
    StructField("_silver_processed_at", TimestampType(), True)
])

# Esquema Oficial: Ocorrências Silver
SCHEMA_OCORRENCIAS_SILVER = StructType([
    StructField("ticket_id", StringType(), False),
    StructField("order_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("severity", StringType(), True),
    StructField("status", StringType(), True),
    StructField("flg_qualidade", BooleanType(), True),
    StructField("_silver_processed_at", TimestampType(), True)
])


## 2. Dicionários de Referência

In [0]:
import pyspark.sql.functions as F

DEFAULT_NULL_TEXT = "NAO INFORMADO"

VALID_UFS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", 
    "PA", "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
]

STATE_MAPPING = {
    "sao paulo": "SP", 
    "são paulo": "SP",
    "sp": "SP",
    "rio de janeiro": "RJ", 
    "rio": "RJ",
    "rj": "RJ",
    "minas gerais": "MG",
    "parana": "PR", 
    "paraná": "PR",
    "santa catarina": "SC", 
    "s. catarina": "SC", 
    "sta catarina": "SC"
}

REGIONAL_MAPPING = {
    'sul': 'S', 's': 'S', 
    'sudeste': 'SE', 'se': 'SE', 
    'norte': 'N', 'n': 'N', 
    'nordeste': 'NE', 'ne': 'NE', 
    'centro-oeste': 'CO', 'co': 'CO'
}

## 3. Funções de Validação e Padronização

In [0]:
def standardize_null_text(col):
    """Padroniza nulos e vazios. Aceita string (nome) ou objeto Column."""
    c = F.col(col) if isinstance(col, str) else col
    return F.when(c.isNull() | (F.trim(c) == "") | (F.lower(c) == "null"), DEFAULT_NULL_TEXT).otherwise(c)

def parse_multi_date(col):
    """Converte strings de data em múltiplos formatos para DateType de forma segura."""
    # Se for string, usamos expr com try_to_date (mais seguro contra exceções)
    if isinstance(col, str):
        return F.coalesce(
            F.expr(f"try_to_date({col}, 'yyyy-MM-dd')"),
            F.expr(f"try_to_date({col}, 'yyyy/MM/dd')"),
            F.expr(f"try_to_date({col}, 'dd-MM-yyyy')"),
            F.expr(f"try_to_date({col}, 'dd/MM/yyyy')"),
            F.expr(f"try_to_date({col}, 'yyyy-MM-dd HH:mm:ss')")
        )
    else:
        return F.coalesce(
            F.to_date(col, 'yyyy-MM-dd'),
            F.to_date(col, 'yyyy/MM/dd'),
            F.to_date(col, 'dd-MM-yyyy'),
            F.to_date(col, 'dd/MM/yyyy')
        )

def clean_numeric(col):
    """Trata inconsistências e garante NULL para valores malformados (ex: 'N/A')."""
    c = F.col(col) if isinstance(col, str) else col
    # Normalizamos o formato (vírgula para ponto)
    cleaned_str = F.regexp_replace(c.cast("string"), ",", ".")
    # Usamos regex para validar se é um número antes de tentar o cast (evita exceções do Spark 3.4)
    # O regex aceita: opcionais de sinal negativo, dígitos, ponto decimal opcional e mais dígitos.
    is_numeric = cleaned_str.rlike(r"^-?\d+\.?\d*$")
    return F.when(is_numeric, cleaned_str.cast("double")).otherwise(F.lit(None).cast("double"))
    
def handle_priority(col):
    c = F.col(col) if isinstance(col, str) else col
    return F.when(c.isNull() | (F.lower(c) == "null"), "low").otherwise(F.lower(c))

def is_valid_email(col):
    c = F.col(col) if isinstance(col, str) else col
    regex = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
    return c.rlike(regex)

def is_valid_state(col):
    c = F.col(col) if isinstance(col, str) else col
    return c.isin(VALID_UFS)

def is_positive_number(col):
    c = F.col(col) if isinstance(col, str) else col
    return (c >= 0) & (c.isNotNull())

def handle_item_status(col):
    c = F.col(col) if isinstance(col, str) else col
    return F.when(c.isNull() | (F.trim(c) == ""), "CANCELADO").otherwise(F.upper(c))

def clean_int(col):
    """Garante a conversão de strings decimais (ex: '3.0') para Inteiro com segurança."""
    c = F.col(col) if isinstance(col, str) else col
    # Converte para double primeiro e depois para int para evitar erro de cast no Spark 3.4+
    return c.cast("double").cast("int")

def parse_multi_timestamp(col):
    """Converte strings de data/hora normalizando o caractere 'T' para espaço."""
    safe_col = f"{col}" if isinstance(col, str) and "." in col else col
        
    # Agora usamos a lógica do regexp_replace direto dentro do SQL (F.expr)
    return F.coalesce(
    F.expr(f"try_to_timestamp(regexp_replace(cast({safe_col} as string), 'T', ' '), 'yyyy-MM-dd HH:mm:ss')"),
    F.expr(f"try_to_timestamp(regexp_replace(cast({safe_col} as string), 'T', ' '), 'dd/MM/yyyy HH:mm:ss')"),
    F.expr(f"try_to_timestamp(regexp_replace(cast({safe_col} as string), 'T', ' '), 'dd/MM/yyyy HH:mm')")
    )



## 4. Premissas e Limitações Adotadas (Documentação Técnica)

### Geral
- **Padronização de Nulos**: Todos os campos de texto categóricos nulos ou vazios são consolidados no termo único **`NAO INFORMADO`**. Isso evita que registros sejam excluídos acidentalmente em filtros de BI.
- **Schema Enforcement**: Escolhemos utilizar `StructType` do Spark em vez de bibliotecas como Pydantic para priorizar a performance em larga escala, evitando custos de serialização entre Spark e Python.

### Clientes
- **Geografia**: Aplicamos uma normalização defensiva que converte variações de nomes de estados para siglas oficiais (UFs). Estados não reconhecidos são marcados como `NAO INFORMADO`.
- **Status**: Tratamento binário onde apenas 'ativo' é considerado como verdadeiro para a flag (`flg_ativo = True`).

### Produtos
- **Preços**: Produtos com `list_price` zerado são sinalizados via `flg_qualidade`, permitindo que o analista decida se deve incluí-los em cálculos financeiros.

### Pedidos
- **Itens (Status)**: Definimos que itens de pedido com status nulo ou em branco na origem são automaticamente classificados como **CANCELADO**. Valores preenchidos são padronizados para maiúsculo.
- **Prioridade**: Definimos a premissa de que registros com `priority` nulo no JSON de origem são classificados como **low**, refletindo o comportamento padrão do sistema de pedidos quando não há urgência sinalizada.
- **Inconsistência Numérica**: O ERP apresenta mistura de separadores decimais (vírgula e ponto). Ajuste via Regex para garantir a integridade dos cálculos de receita.
- **Integridade de Valores**: Marcamos como baixa qualidade (`flg_qualidade = False`) pedidos onde o valor líquido é superior ao valor bruto, o que indicaria um erro de cálculo de impostos/descontos.
- **Duplicidade de Itens**: Registros com o mesmo `order_id` que possuam `item_seq` ou `product_code` repetidos são sinalizados como baixa qualidade para evitar distorção em métricas de volume e SKU.


### Vendedores
- **Status**: Vendedores com status nulo ou em branco na origem são considerados **ativos** (`flg_ativo = True`).

### Regiões
- **Padronização Geográfica**: Regional e Estado são padronizados para siglas (ex: `S`, `SP`) para garantir unicidade e joins precisos. Registros duplicados após padronização são removidos.
### Performance
- **Z-Ordering**: Aplicamos Z-Order nas colunas de busca frequente para otimizar o Data Skipping do Delta Lake.